In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from google.colab import userdata
token = userdata.get('HF_TOKEN')

In [3]:
%%capture
from unsloth import FastLanguageModel
import torch

# model_id = "unsloth/Qwen3-4B-Instruct-2507" # 例如 "huggingface/llama-3-8b-lora"
model_id = "TicklingShell/tsa-qwen-lora-6eps"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    load_in_4bit = True, # 如果显存紧张，继续用4bit
)
FastLanguageModel.for_inference(model)

In [5]:
from unsloth.chat_templates import get_chat_template
from datasets import load_from_disk

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

def filter_invalid(dataset):
    n_original = dataset.num_rows
    filtered = dataset.filter(lambda x: x['目标数量'] != -1, keep_in_memory=True, load_from_cache_file=False)

    n_invalid = n_original - filtered.num_rows
    print(f"Removed {n_invalid} invalid rows. {filtered.num_rows} rows remaining.")

    return filtered

dataset = load_from_disk("./tsa-train")
dataset = filter_invalid(dataset)

Filter:   0%|          | 0/6300 [00:00<?, ? examples/s]

Removed 25 invalid rows. 6275 rows remaining.


In [6]:
SYSTEM_PROMPT = """
你是一名精通目标情感分析（TSA）的数据科学家。你的核心任务是分析用户评论，提取被评价的目标实体，并评估其情感倾向。在工作中，你必须严格遵守以下输出格式规范：

【全局输出格式规范】
你必须且只能输出一个标准的 JSON 对象，严禁包裹任何 Markdown 语法标记（如 ```json）。该对象必须严格包含以下四个中文键名，绝对不得出现任何英文键名：
- "目标": 列表（List），由你提取出的标准实体名称字符串组成。
- "标签": 列表（List），与"目标"列表中的实体一一对应，填充情感倾向数字（1代表正向，0代表中性，-1代表负向）。
- "理由": 列表（List），与"目标"列表中的实体一一对应，详细阐述该目标获得此情感标签的具体依据。

【行为边界】
你只需输出符合上述规范的 JSON 数据，不得输出任何前导词、解释性文字或后续总结。确保 JSON 格式绝对合法。
"""

In [7]:
USER_PROMPT = """
【任务指令】
请对后面的评论执行深度“目标情感分析”（TSA）任务。请严格按照以下步骤进行分析：
1. 实体提取与边界划定：扫描全文，精准提取出所有被评价的目标实体（注意划定文本边界，并挖掘出未明说但实际被评价的“隐含实体”）。
2. 文本修辞与语境辨析：仔细分辨评论中对各实体的评价是否存在反讽、隐喻、夸张等情况，还原用户的真实表达意图。
3. 情感倾向处理：在排除修辞干扰后，判断最终真实情感倾向（1代表正向，0代表中性，-1代表负向）。

【输出格式约束】
你必须输出一个标准的JSON对象，且必须严格包含以下中文键名，不得出现任何英文键名：
- "目标": 列表，由你提取出的标准实体名称组成。
- "标签": 列表，与"目标"列表一一对应的情感倾向数字（1, 0, 或 -1）。
- "理由": 列表，与"目标"列表一一对应，详细阐述该目标获得此标签的具体依据。

【示例】
评论：这手机拍照真“清晰”，大白天拍人能拍出鬼影来，不过续航确实顶，用了一天还有一半电。

输出JSON：
{
    "思维链": "1. 实体提取：显式实体有‘续航’（用了一天还有电）。‘拍照真清晰’中提取出隐含实体‘拍照’。 2. 语境辨析：‘拍照真清晰’加了双引号，且后文提到‘拍出鬼影’，判定属于强烈的反讽修辞，真实意图是极度不满；‘续航确实顶’为夸张赞美，无反讽。 3. 情感处理：‘拍照’排除反讽干扰后为负向（-1），‘续航’为正向（1）。",
    "目标": ["拍照", "续航"],
    "标签": [-1, 1],
    "理由": [
        "评论使用反讽手法，表面夸清晰实际指出大白天拍出鬼影，对拍照功能极度不满。",
        "评论直言续航确实顶，并用具体数据（用了一天还有一半电）证实了对续航的强烈认可。"
    ]
}

【待分析评论】
评论：
"""

In [8]:
def tsa(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT + f"{example['评论']}\n输出JSON："}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True, # Must add for generation
    )

    output = model.generate(
        **tokenizer(text, return_tensors = "pt").to("cuda"),
        max_new_tokens = 1000, # Increase for longer outputs!
        temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
        use_cache = True,
    )

    output_ids = output[0][len(tokenizer(text, return_tensors = "pt").to("cuda").input_ids[0]):].tolist()
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

def calculate_f1(y_true, y_pred):
    """
    y_true: 列表，每个元素是 set()，如 [{"A", "B"}, {"C"}]
    y_pred: 列表，每个元素是 set()，如 [{"A", "D"}, {"C"}]
    """
    # 初始化用于 Micro-F1 的全局计数器
    total_tp = 0
    total_fp = 0
    total_fn = 0

    # 用于存储每条样本 F1 的列表，用于计算 Macro-F1
    sample_f1_list = []

    for gold, pred in zip(y_true, y_pred):
        # 集合运算
        tp = len(gold & pred)       # 预测对的
        fp = len(pred - gold)       # 多预测的（幻觉）
        fn = len(gold - pred)       # 没预测到的（漏掉）

        # --- 计算 Micro 所需的全局累加 ---
        total_tp += tp
        total_fp += fp
        total_fn += fn

        # --- 计算当前样本的 F1 (用于 Macro) ---
        p = tp / len(pred) if len(pred) > 0 else 0
        r = tp / len(gold) if len(gold) > 0 else 0
        s_f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0
        sample_f1_list.append(s_f1)

    # 1. 计算 Micro-F1
    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = (2 * micro_p * micro_r) / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0

    # 2. 计算 Macro-F1 (样本平均)
    macro_f1 = sum(sample_f1_list) / len(sample_f1_list) if len(sample_f1_list) > 0 else 0

    return {
        "Micro-F1": round(micro_f1, 4),
        "Macro-F1": round(macro_f1, 4),
        "Micro-Precision": round(micro_p, 4),
        "Micro-Recall": round(micro_r, 4)
    }

def calculate_conditional_f1(y_true_combined, y_pred_combined):
    """
    y_true_combined: list of set([(target, sentiment), ...])
    y_pred_combined: list of set([(target, sentiment), ...])
    """
    cond_true = []
    cond_pred = []

    for gold_set, pred_set in zip(y_true_combined, y_pred_combined):
        # 找出 gold 中的所有 target
        gold_targets = {t for t, s in gold_set}
        # 找出 pred 中的所有 target
        pred_targets = {t for t, s in pred_set}

        # 只有当 target 在两边都出现时，才把它们的情感标签放进对比池
        common_targets = gold_targets & pred_targets

        for t in common_targets:
            # 找到 gold 中该 target 对应的情感
            g_s = [s for target, s in gold_set if target == t][0]
            # 找到 pred 中该 target 对应的情感
            p_s = [s for target, s in pred_set if target == t][0]

            # 为了复用之前的 calculate_f1，我们包装成集合形式
            cond_true.append({g_s})
            cond_pred.append({p_s})

    # 调用你现有的 calculate_f1 函数
    return calculate_f1(cond_true, cond_pred)

In [ ]:
import json
from tqdm import tqdm

val_split = dataset.select(range(5000, 6200))

# 分别存储仅目标，以及目标+情感的集合
golds_target = []
preds_target = []

golds_combined = []
preds_combined = []

for example in tqdm(val_split, "Eval"):
    output = tsa(example)

    # --- 处理 Gold (真值) ---
    target_set = set(example['目标'])
    label_set = set(zip(example['目标'], example['标签']))

    golds_target.append(target_set)
    golds_combined.append(label_set)

    # --- 处理 Prediction (预测值) ---
    try:
        json_body = json.loads(output)
        p_target = set(json_body['目标'])
        # 即使模型输出的列表长度不一，zip 也会按最短的处理，或者报错
        p_combined = set(zip(json_body['目标'], json_body['标签']))

        preds_target.append(p_target)
        preds_combined.append(p_combined)
    except:
        # 解析失败时，两组预测都存入空集
        preds_target.append(set())
        preds_combined.append(set())

Eval:   0%|          | 1/975 [01:12<19:38:12, 72.58s/it]


KeyboardInterrupt: 

### 优化方案：Batch 推理与鲁棒解析
我们将推理过程重构为批量处理，并引入正则表达式解析以应对 JSON 外多余的描述文字。

In [ ]:
import re
import json
from tqdm import tqdm
import torch

# 1. 确保 Tokenizer 配置正确（生成任务必须左补齐）
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def robust_json_parse(text):
    """使用正则提取 JSON 部分，提高解析成功率"""
    try:
        # 查找第一个 { 和最后一个 }
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            data = json.loads(match.group())
            return data.get('目标', []), data.get('标签', [])
        return [], []
    except:
        return [], []

# 2. 准备数据子集
val_samples = dataset.select(range(5000, 6200))
batch_size = 32 # 根据 T4 显存建议设为 4-8

all_preds_target = []
all_preds_combined = []
all_golds_target = []
all_golds_combined = []

# 3. Batch 推理循环
for i in tqdm(range(0, len(val_samples), batch_size), desc="Batch Inference"):
    batch = val_samples.select(range(i, min(i + batch_size, len(val_samples))))

    # 构建 Prompt Batch
    prompts = [tokenizer.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT + f"{ex['评论']}\n输出JSON："}
    ], tokenize=False, add_generation_prompt=True) for ex in batch]

    inputs = tokenizer(prompts, padding=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1000,
            use_cache=True,
            temperature = 0.7, top_p = 0.8, top_k = 20,
        )

    # 解码并提取结果
    input_len = inputs.input_ids.shape[1]
    for j, ex in enumerate(batch):
        raw_output = tokenizer.decode(generated_ids[j][input_len:], skip_special_tokens=True)
        p_targets, p_labels = robust_json_parse(raw_output)

        # 存储预测值
        try:
            p_target_set = set(p_targets)
            p_combined_set = set(zip(p_targets, p_labels))

            all_preds_target.append(p_target_set)
            all_preds_combined.append(p_combined_set)
        except:
            all_preds_target.append(set())
            all_preds_combined.append(set())

        # 存储真值
        all_golds_target.append(set(ex['目标']))
        all_golds_combined.append(set(zip(ex['目标'], ex['标签'])))

Batch Inference:  21%|██        | 8/38 [11:30<40:32, 81.09s/it]

In [ ]:
print("Result for Model w/o Augmentation: ")
print(f"Target Extraction F1: {calculate_f1(all_golds_target, all_preds_target)}")
print(f"Combined TSA F1: {calculate_f1(all_golds_combined, all_preds_combined)}")
print(f"Conditional F1: {calculate_conditional_f1(all_golds_combined, all_preds_combined)}")

## Archived Analysis

In [10]:
print("Result for Model w/o Reasons: ")
print(f"Target Extraction F1: {calculate_f1(all_golds_target, all_preds_target)}")
print(f"Combined TSA F1: {calculate_f1(all_golds_combined, all_preds_combined)}")
print(f"Conditional F1: {calculate_conditional_f1(all_golds_combined, all_preds_combined)}")

Result for Model w/o Reasons: 
Target Extraction F1: {'Micro-F1': 0.6051, 'Macro-F1': 0.5815, 'Micro-Precision': 0.5664, 'Micro-Recall': 0.6494}
Combined TSA F1: {'Micro-F1': 0.5501, 'Macro-F1': 0.5294, 'Micro-Precision': 0.5151, 'Micro-Recall': 0.5903}
Conditional F1: {'Micro-F1': 0.9091, 'Macro-F1': 0.9091, 'Micro-Precision': 0.9091, 'Micro-Recall': 0.9091}


In [ ]:
print("Result for Base Model: ")
print(f"Target Extraction F1: {calculate_f1(all_golds_target, all_preds_target)}")
print(f"Combined TSA F1: {calculate_f1(all_golds_combined, all_preds_combined)}")
print(f"Conditional F1: {calculate_conditional_f1(all_golds_combined, all_preds_combined)}")

Result for Base Model: 
Target Extraction F1: {'Micro-F1': 0.4812, 'Macro-F1': 0.4531, 'Micro-Precision': 0.4539, 'Micro-Recall': 0.512}
Combined TSA F1: {'Micro-F1': 0.4062, 'Macro-F1': 0.3789, 'Micro-Precision': 0.3834, 'Micro-Recall': 0.432}
Conditional F1: {'Micro-F1': 0.8443, 'Macro-F1': 0.8443, 'Micro-Precision': 0.8443, 'Micro-Recall': 0.8443}


In [ ]:
# 计算仅实体的提取效果
print("Target Extraction F1:")
print(calculate_f1(golds_target, preds_target))

# 计算实体+情感都对齐的效果
print("Combined TSA F1:")
print(calculate_f1(golds_combined, preds_combined))

print("Conditional F1:")
print(calculate_conditional_f1(golds_combined, preds_combined))

Target Extraction F1:
{'Micro-F1': 0, 'Macro-F1': 0.0, 'Micro-Precision': 0.0, 'Micro-Recall': 0.0}
Combined TSA F1:
{'Micro-F1': 0, 'Macro-F1': 0.0, 'Micro-Precision': 0.0, 'Micro-Recall': 0.0}
Conditional F1:
{'Micro-F1': 0, 'Macro-F1': 0, 'Micro-Precision': 0, 'Micro-Recall': 0}
